<a href="https://colab.research.google.com/github/mattnewdavid/dm-medgemma-finetune/blob/main/medgemma_dm_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q trl accelerate peft bitsandbytes transformers datasets

In [ ]:
import torch
import os
from transformers import (Pipeline, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig)
from peft import LoraConfig
from trl import SFTTrainer
from datasets import load_dataset
from google.colab import userdata


In [ ]:
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
dataset = load_dataset("abdelhakimDZ/diabetes_QA_dataset")
print(dataset)
print(dataset['train'][0])

In [ ]:
def format_dataset(example):
  prompt = f"""You are a medical assistant specializing in diabetes. Your task is to answer
  Question: {example['question']}
  Answer: """
  example['text'] = prompt + " " + example['answer']
  return example

dataset = dataset.map(format_dataset, remove_columns=["question","answer"])

In [ ]:
print(dataset["train"][0])
dataset = dataset["train"].train_test_split(test_size=0.2)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
model_id = "google/medgemma-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
def tokenize_function(example):

    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

    # Fix for Gemma training
    if "token_type_ids" not in tokens:
        tokens["token_type_ids"] = [0] * len(tokens["input_ids"])

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
train_dataset = train_dataset.map(tokenize_function, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(tokenize_function, remove_columns=eval_dataset.column_names)

In [ ]:
print(train_dataset[0])

In [ ]:
lora_config = LoraConfig(
    r=8,

    target_modules=["q_proj", "v_proj"],

    task_type="CAUSAL_LM"
)

In [ ]:
training_args = TrainingArguments(
    output_dir="medgemma-dm",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_32bit",
    report_to="none"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    args=training_args,
)

In [ ]:
import torch
from transformers import DataCollatorWithPadding

class CustomDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        # Use the parent class's call method for padding and tensor conversion
        # The return_tensors argument is passed during initialization of DataCollatorWithPadding, not its __call__ method.
        batch = super().__call__(features)

        # Explicitly ensure 'token_type_ids' are present and are tensors of zeros
        if "token_type_ids" not in batch:
            # If not present (e.g., from original tokenizer not generating them)
            # or dropped by DataCollatorWithPadding for some reason, add them.
            # They should have the same shape as input_ids.
            batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])
        elif not torch.is_tensor(batch["token_type_ids"]):
            # If it's a list or numpy array (unlikely with return_tensors="pt"), convert to tensor
            batch["token_type_ids"] = torch.tensor(batch["token_type_ids"], dtype=torch.long)

        return batch

# Ensure 'token_type_ids' is in the tokenizer's model_input_names for general compatibility,
# although our custom collator explicitly handles it.
if "token_type_ids" not in tokenizer.model_input_names:
    tokenizer.model_input_names.append("token_type_ids")

data_collator = CustomDataCollator(tokenizer=tokenizer)

# Re-initialize trainer with the custom data_collator
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    args=training_args,
    data_collator=data_collator,
)
trainer.train()